In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt
import ssm
from pathlib import Path
import pickle
import copy
import warnings
warnings.filterwarnings('ignore')

from notebooks.imports import *
from config import dir_config
from src.utils.glm_hmm_utils import global_fit
from src.utils.glm_hmm_utils_cv import session_wise_fit_cv

## Configuration

Fits **4 PD groups** independently (matching the DDM notebook 3.40 structure):
- Tremor-dominant OFF medication
- Tremor-dominant ON medication
- Bradykinesia-dominant OFF medication
- Bradykinesia-dominant ON medication

Each group is fit with K = 1–4 states. K is **not inherited from HC** — the optimal K for each group is determined by its own cross-validation curve. Final consensus K is chosen in notebook 4.40.

**Session handling:** ON and OFF sessions are always separate — HMM chains never cross session boundaries.

In [ ]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir = Path(processed_dir, 'glm_hmm_models')

MODEL_NAME = 'prior_model_color_1back'

with open(Path(glm_hmm_dir, f'{MODEL_NAME}_config.pkl'), 'rb') as f:
    feature_config = pickle.load(f)

CURRENT_TRIAL_FEATURES = feature_config['current_trial_features']
PREV_TRIAL_FEATURES    = feature_config['prev_trial_features']
N_TRIALS_BACK          = feature_config['n_trials_back']
MODEL_FEATURES         = feature_config['model_features']
INPUT_DIM              = feature_config['input_dim']

GROUPS = {
    'tremor_off': feature_config['session_ids']['tremor_off'],
    'tremor_on':  feature_config['session_ids']['tremor_on'],
    'brady_off':  feature_config['session_ids']['brady_off'],
    'brady_on':   feature_config['session_ids']['brady_on'],
}
GROUP_LABELS = {
    'tremor_off': 'Tremor OFF',
    'tremor_on':  'Tremor ON',
    'brady_off':  'Brady OFF',
    'brady_on':   'Brady ON',
}

STATE_RANGE    = np.arange(1, 5)
N_INITS        = 20
N_ITERS_GLOBAL = 2500
N_ITERS_CV     = 1000
K_FOLDS        = 5
TOLERANCE      = 1e-4

print('Model features:', MODEL_FEATURES)
print('Input dim:', INPUT_DIM)
print('Groups and session counts:')
for g, ids in GROUPS.items():
    print(f'  {GROUP_LABELS[g]}: {len(ids)} sessions')

## Load Behavioral Data

In [ ]:
data = pd.read_csv(Path(glm_hmm_dir, f'{MODEL_NAME}_data.csv'))
data.choice  = data.choice.fillna(-1).astype(int)
data.outcome = data.outcome.fillna(-1).astype(int)
print('Behavioral data shape:', data.shape)

## Shared Data Processing Functions

In [ ]:
def extract_previous_trial_data(session_data, valid_idx, first_trial):
    n_trials = session_data.shape[0] - first_trial
    prev_data = {}
    signed_coherence = session_data.signed_coherence.values / 100
    choice = session_data.choice.values * 2 - 1  # recode -1/+1

    for var in PREV_TRIAL_FEATURES:
        prev_data[var] = np.empty((n_trials, N_TRIALS_BACK))

    for i in range(first_trial, session_data.shape[0]):
        valid_before = valid_idx[valid_idx < i][-N_TRIALS_BACK:]
        padded = np.pad(valid_before, (N_TRIALS_BACK - len(valid_before), 0),
                        'constant', constant_values=0)
        for var in PREV_TRIAL_FEATURES:
            var_col = var[5:] if var.startswith('prev_') else var
            if var_col == 'coherence':
                vals = signed_coherence
            elif var_col == 'choice_coherence':
                vals = choice * signed_coherence
            elif var_col == 'choice':
                vals = choice  # use ±1 recode (symmetric, zero-centered)
            else:
                vals = session_data[var_col].values
            prev_data[var][i - first_trial] = vals[padded]
    return prev_data


def prepare_input_data(session_data, valid_idx, first_trial):
    n_trials = session_data.shape[0] - first_trial
    X = np.ones((1, n_trials, INPUT_DIM))
    for idx, feat in enumerate(CURRENT_TRIAL_FEATURES):
        if feat == 'normalized_stimulus':
            X[0, :, idx] = session_data.signed_coherence.values[first_trial:] / 100
        elif feat == 'bias':
            X[0, :, idx] = 1.0
        else:
            X[0, :, idx] = session_data[feat].values[first_trial:]
    prev_data = extract_previous_trial_data(session_data, valid_idx, first_trial)
    col_idx = len(CURRENT_TRIAL_FEATURES)
    for var in PREV_TRIAL_FEATURES:
        for n in range(N_TRIALS_BACK):
            X[0, :, col_idx] = prev_data[var][:, n]
            col_idx += 1
    return list(X)


def process_sessions(data, session_ids, seed=42):
    npr.seed(seed)
    inputs_list, choices_list, masks_list = [], [], []
    unnorm_list, rt_list, color_list = [], [], []
    used_ids = []

    for session_id in session_ids:
        sess = data[data['session_id'] == session_id].reset_index(drop=True)
        if len(sess) == 0:
            print(f'  WARNING: {session_id} not found in data — skipping.')
            continue
        valid_idx = np.where(sess.outcome >= 0)[0]
        if len(valid_idx) < N_TRIALS_BACK + 5:
            print(f'  WARNING: {session_id} — only {len(valid_idx)} valid trials — skipping.')
            continue
        first_trial = valid_idx[N_TRIALS_BACK - 1] + 1

        inputs = prepare_input_data(sess, valid_idx, first_trial)
        choices = sess.choice.values[first_trial:].astype(int)
        invalid_idx = np.where(sess.outcome.values[first_trial:] < 0)[0]
        if invalid_idx.size > 0:
            choices[invalid_idx] = np.random.choice([0, 1], size=invalid_idx.size)
        mask = np.ones_like(choices, dtype=bool)
        mask[invalid_idx] = False

        inputs_list += inputs
        choices_list.append(choices.reshape(-1, 1))
        masks_list.append(mask.reshape(-1, 1))
        rt_list.append(sess.reaction_time.values[first_trial:].reshape(-1, 1))
        color_list.append(sess.color.values[first_trial:].reshape(-1, 1))
        used_ids.append(session_id)

    # Fixed, zero-preserving scaling (signed_coherence / 100) is applied in
    # prepare_input_data / extract_previous_trial_data; prev_choice is ±1.
    # We deliberately do NOT z-score per session:
    #   - mean-centering would move physical coherence=0 off zero and fold the
    #     prior-induced coherence-sampling skew into the bias regressor (the
    #     primary readout of prior integration);
    #   - per-session std scaling would make stimulus/history weights
    #     incomparable across sessions and groups (HC vs PD ON/OFF).
    # The /100 scale is identical across all sessions/groups and keeps current
    # and previous coherence on the same units. unnorm_list is retained for API
    # compatibility (now identical to inputs_list).
    unnorm_list = copy.deepcopy(inputs_list)

    return inputs_list, choices_list, masks_list, unnorm_list, rt_list, color_list, used_ids

## Group Fitting Function

For each group:
1. Build design matrices from that group's sessions
2. Global fit: K = 1–4, 20 random initializations, 2500 EM iterations
3. Session-wise 5-fold CV: K = 1–4, initialized from best global params
4. Posterior inference using best global model (state ordering by `prior_strength` weight)
5. Return complete result dict

In [ ]:
def fit_group(group_name, session_ids):
    print(f'\n========== {GROUP_LABELS[group_name]} ({len(session_ids)} sessions) ==========')
    inputs, choices, masks, unnorm, rts, colors, used_ids = process_sessions(data, session_ids)
    n_used = len(used_ids)
    print(f'  Using {n_used} sessions after quality checks.')
    if n_used == 0:
        print('  No valid sessions — skipping.')
        return None

    print(f'  Global fit: K={STATE_RANGE}, {N_INITS} inits, {N_ITERS_GLOBAL} iters...')
    models_global, lls_global = global_fit(
        choices, inputs, masks,
        state_range=STATE_RANGE,
        n_initializations=N_INITS,
        n_iters=N_ITERS_GLOBAL,
        tolerance=TOLERANCE,
    )

    init_params = {'glm_weights': {}, 'transition_matrices': {}}
    for k in STATE_RANGE:
        best_idx = np.argmax([lls_global[k][i][-1] for i in range(N_INITS)])
        init_params['glm_weights'][k]        = models_global[k][best_idx].observations.params
        init_params['transition_matrices'][k] = models_global[k][best_idx].transitions.params
        print(f'    K={k}: best LL={lls_global[k][best_idx][-1]:.3f}')

    print(f'  Session-wise {K_FOLDS}-fold CV...')
    models_cv, train_ll, test_ll = session_wise_fit_cv(
        choices, inputs, masks,
        n_sessions=n_used,
        init_params=init_params,
        k_folds=K_FOLDS,
        state_range=STATE_RANGE,
        n_iters=N_ITERS_CV,
        tolerance=TOLERANCE,
    )

    # State ordering by the prior-integration regressor (descending):
    # 'color' if the model includes it, otherwise 'bias' — which, because
    # direction is pre-normalized so choice=1 = toward the prior, is itself the
    # directional-prior readout. State 1 = most prior-engaged.
    order_feat = 'color' if 'color' in MODEL_FEATURES else 'bias'
    order_col = MODEL_FEATURES.index(order_feat)
    posteriors_by_k = {}
    for k in STATE_RANGE:
        best_idx   = np.argmax([lls_global[k][i][-1] for i in range(N_INITS)])
        best_model = models_global[k][best_idx]

        # Negate: SSM params predict P(y=0); negate → P(y=1=toward prior direction)
        w_raw = -best_model.observations.params.squeeze(1)
        T_raw =  best_model.transitions.params[0]

        state_order = np.argsort(w_raw[:, order_col])[::-1]
        w_ord = w_raw[state_order]
        T_ord = T_raw[np.ix_(state_order, state_order)]

        smoothed_list, viterbi_list = [], []
        for idx_sess in range(n_used):
            post = best_model.expected_states(
                choices[idx_sess], input=inputs[idx_sess], mask=masks[idx_sess]
            )[0][:, state_order]
            smoothed_list.append(post)

            vit_raw = best_model.most_likely_states(
                choices[idx_sess], input=inputs[idx_sess], mask=masks[idx_sess]
            )
            remap = {old: new for new, old in enumerate(state_order)}
            viterbi_list.append(np.array([remap[s] for s in vit_raw]))

        occupancy = np.array([[np.mean(v == s) for s in range(k)]
                               for v in viterbi_list])

        posteriors_by_k[k] = {
            'state_order': state_order.tolist(),
            'weights':     w_ord,
            'transition':  T_ord,
            'smoothed':    smoothed_list,
            'viterbi':     viterbi_list,
            'occupancy':   occupancy,
        }

    session_data_dict = {}
    for idx, sid in enumerate(used_ids):
        df = {
            'choices': choices[idx].ravel(),
            'stimulus': unnorm[idx][:, MODEL_FEATURES.index('normalized_stimulus')],
            'color':  colors[idx].ravel(),
            'mask':   masks[idx].ravel(),
        }
        for i, feat in enumerate(MODEL_FEATURES):
            df[feat] = inputs[idx][:, i]
        session_data_dict[sid] = pd.DataFrame(df)

    result = {
        'group':            group_name,
        'group_label':      GROUP_LABELS[group_name],
        'used_session_ids': used_ids,
        'config':           feature_config,
        'state_range':      STATE_RANGE.tolist(),
        'global': {'models': models_global, 'lls': lls_global, 'init_params': init_params},
        'cv':     {'models': models_cv, 'train_ll': train_ll, 'test_ll': test_ll},
        'posteriors_by_k': posteriors_by_k,
        'data':             session_data_dict,
    }

    for k in STATE_RANGE:
        occ     = posteriors_by_k[k]['occupancy']
        mean_ll = np.nanmean(test_ll[:, STATE_RANGE.tolist().index(k), :])
        occ_str = '  '.join([f'S{s+1}={occ[:,s].mean():.2f}' for s in range(k)])
        print(f'    K={k}: test_LL={mean_ll:.4f}  occupancy: {occ_str}')

    return result

## Fit All 4 PD Groups

Run cells sequentially — each group takes ~10–30 minutes depending on hardware.

In [ ]:
results = {}
results['tremor_off'] = fit_group('tremor_off', GROUPS['tremor_off'])

In [ ]:
results['tremor_on'] = fit_group('tremor_on', GROUPS['tremor_on'])

In [ ]:
results['brady_off'] = fit_group('brady_off', GROUPS['brady_off'])

In [ ]:
results['brady_on'] = fit_group('brady_on', GROUPS['brady_on'])

## Cross-Group Model Selection Preview

Plot test LL vs K for all 4 PD groups on one figure. The consensus K is determined formally in notebook 4.40 alongside the HC curve.

In [ ]:
GROUP_COLORS = {
    'tremor_off': '#d62728',
    'tremor_on':  '#ff9896',
    'brady_off':  '#2ca02c',
    'brady_on':   '#98df8a',
}

fig, ax = plt.subplots(figsize=(6, 4))
for grp_key, res in results.items():
    if res is None:
        continue
    test_ll = res['cv']['test_ll']  # (n_sessions, n_states, k_folds)
    mean_ll = np.nanmean(test_ll, axis=(0, 2))
    sem_ll  = np.nanstd(test_ll, axis=(0, 2)) / np.sqrt(test_ll.shape[0] * test_ll.shape[2])
    ax.errorbar(STATE_RANGE, mean_ll, yerr=sem_ll, fmt='o-',
                color=GROUP_COLORS[grp_key], label=GROUP_LABELS[grp_key],
                capsize=3, lw=2, ms=6)

ax.set_xlabel('Number of states (K)', fontsize=13)
ax.set_ylabel('Mean test log-likelihood per trial', fontsize=13)
ax.set_title('PD groups — model selection preview', fontsize=13)
ax.set_xticks(STATE_RANGE)
ax.legend(fontsize=9)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(Path(glm_hmm_dir, f'{MODEL_NAME}_pd_model_selection_preview.pdf'), bbox_inches='tight')
plt.show()

## Save All PD Results

In [ ]:
out_path = Path(glm_hmm_dir, f'{MODEL_NAME}_pd_groups.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)

print(f'Saved PD group results to: {out_path}')
for grp_key, res in results.items():
    if res is None:
        print(f'  {GROUP_LABELS[grp_key]}: FAILED')
    else:
        n = len(res['used_session_ids'])
        print(f'  {GROUP_LABELS[grp_key]}: {n} sessions, K=1-4 fitted')